1. Start with the existing pretrained checkpoint

Take 100–500 real sequences and do a counterfactual age sweep. Keep the entire patient's events and timestamps fixed, but evaluate the model at ages such as

$$ a\in\{1,5,10,15\}. $$

Save at every age:

$$ \Delta\alpha(a),\quad \alpha(a),\quad \log w_{ij}(a),\quad QK^\top/\sqrt d,\quad A_{ij},\quad h_{\text{final}},\quad \text{output logits}. $$

This gives you a localization test:

Observation                                     	Likely problem
\( \Delta\alpha(a)\) barely changes	                Age encoder / Fourier / MLP never learned age

\(\Delta\alpha\) changes, \(\log w\) doesn't	    Temporal basis or \(\tau\) scaling is killing it

\(\log w\) changes, attention barely changes	    Kernel contribution is too small relative to \(QK\)

Attention changes, representation doesn't	        Residual/content pathway is ignoring it

Representation changes, prediction doesn't	        Pretraining objective does not require the information

Everything changes sensibly	Mechanism works;        downstream task/data may simply not benefit

The saved checkpoints I have, We can validate all, or find the best/ correct one:
From newest to oldest:
1. /home/suraj/Git/Age-conditioned-pediatric-EHR/model_new/run/kernel_s0_072420260946/best.pt
2. /home/suraj/Git/Age-conditioned-pediatric-EHR/checkpoints/age_real_full_20260622/epoch_010.pt
3. /home/suraj/Git/Age-conditioned-pediatric-EHR/checkpoints/age_real_full0504_2130/epoch_010.pt

## Checkpoint 1 — `model_new` kernel arm

Path: `model_new/run/kernel_s0_072420260946/best.pt` (symlink to `epoch_008.pt`).

This is the DKM pretrain model in `model_new/`, arm `kernel`. Age reaches the model on two routes; this test **isolates the kernel route only**:

- **Held fixed:** codes, timestamps, padding, and `demographics` (including the standardized age channel).
- **Swept:** `age_years`, the tensor that feeds `ψ(a) → Δα(a) → log w`.

At every counterfactual age we record the chain

`Δα(a) → α(a) → log w_ij(a) → QKᵀ/√d → A_ij → h → logits`

and stop at the first link that does not move. Layer-0 `QKᵀ/√d` is computed from frozen code embeddings, so it must be age-invariant; a change there is a wiring bug, not a finding.

MIMIC pretraining support starts at ~16.6 y, so `{1, 5, 10, 15}` is extrapolation. The Δα cell also evaluates a few adult ages so we do not confuse “pediatric grid is flat” with “the encoder never learned age”.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import torch
import torch.nn.functional as F

REPO = Path("/home/suraj/Git/Age-conditioned-pediatric-EHR")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from model_new.data import TensorizedPretrainDataset, tau_from_timestamps
from model_new.encoder import build_pair_mask
from model_new.eval_pretrain import build_model, make_val_loader, model_kwargs_from_config

RUN_DIR = REPO / "model_new/run/kernel_s0_072420260946"
CKPT_PATH = RUN_DIR / "best.pt"
AGES = (1.0, 5.0, 10.0, 15.0)
ADULT_AGES = (18.0, 40.0, 65.0, 80.0)
N_SEQ = 128
BATCH_SIZE = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = json.loads((RUN_DIR / "config.json").read_text())
shared = model_kwargs_from_config(cfg)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print(f"checkpoint : {CKPT_PATH}")
print(f"  arm={ckpt.get('arm')}  epoch={ckpt.get('epoch')}  seed={ckpt.get('seed')}")
print(f"  tau_max file={ckpt.get('tau_max')}  config={shared['tau_max']}")
if ckpt.get("arm") != "kernel":
    raise RuntimeError(f"expected arm='kernel', got {ckpt.get('arm')!r}")

model = build_model(shared, "kernel")
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(device).eval()
print(f"  device={device}  model.tau_max={float(model.tau_max):.6f}")
print(f"  n_layers={model.n_layers}  n_heads={model.n_heads}  s={model.s}")

ds = TensorizedPretrainDataset(
    REPO / shared["tensorized_dir"] / "val",
    REPO / shared["vocab_path"],
    max_seq_len=shared["max_seq_len"],
)
print(f"  val windows={len(ds):,}  using first {N_SEQ}")
torch.set_grad_enabled(False)


def to_device(batch):
    return {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}


def broadcast_age(batch, age):
    """Replace kernel-site ages only. Demographics (R1) stay at the true values."""
    out = dict(batch)
    mask = batch["attention_mask"].to(dtype=batch["age_years"].dtype)
    out["age_years"] = torch.full_like(batch["age_years"], float(age)) * mask
    return out


def take_sequences(n):
    rows = []
    for batch in make_val_loader(ds, BATCH_SIZE, 0, shared["race_encoding"]):
        rows.append(batch)
        if sum(b["lengths"].shape[0] for b in rows) >= n:
            break
    # trim the last batch so we have exactly n rows
    kept, need = [], n
    for batch in rows:
        bsz = batch["lengths"].shape[0]
        if bsz <= need:
            kept.append(batch)
            need -= bsz
        else:
            kept.append({k: (v[:need] if isinstance(v, torch.Tensor) else v) for k, v in batch.items()})
            need = 0
        if need == 0:
            break
    return kept


In [ ]:
@torch.no_grad()
def site_curve(site, ages):
    a = torch.tensor(list(ages), dtype=torch.float32, device=device)
    delta = site.age(a)                  # Δα(a)  [A, s]
    alpha = site.alpha_base + delta      # α(a)   [A, s]
    return delta.cpu(), alpha.cpu()


def curve_span(delta):
    """Max pairwise L2 over the age grid. 0 ⇒ Δα does not depend on a."""
    x = delta.float()
    return float(torch.cdist(x, x).max())


print("generator last-layer (W2) and α_base — still-zero W2 means Δα never left init")
for name, site in model.kernel_sites():
    W2 = site.age.generator.mlp[-1].weight.detach()
    print(f"  {name:16s}  ||W2||_F={float(W2.norm()):.4e}  "
          f"α_base={site.alpha_base.detach().cpu().tolist()}")

print("\nΔα(a) and α(a) on the pediatric sweep ages", AGES)
ckpt1_alpha = {}
for name, site in model.kernel_sites():
    delta, alpha = site_curve(site, AGES)
    ckpt1_alpha[name] = {"ages": AGES, "delta_alpha": delta, "alpha": alpha}
    print(f"\n  {name}")
    print(f"    Δα rows (one per age, s={delta.shape[-1]}):")
    for a, d, al in zip(AGES, delta, alpha):
        print(f"      a={a:5.1f}  Δα={d.tolist()}  ||Δα||={float(d.norm()):.4e}  "
              f"α={al.tolist()}")
    print(f"    max pairwise ||Δα(a)-Δα(a')|| = {curve_span(delta):.4e}")

print("\nSame Δα span on adult ages (in-support for this checkpoint)", ADULT_AGES)
for name, site in model.kernel_sites():
    delta, _ = site_curve(site, ADULT_AGES)
    print(f"  {name:16s}  max pairwise ||Δα|| = {curve_span(delta):.4e}")
    for a, d in zip(ADULT_AGES, delta):
        print(f"    a={a:5.1f}  ||Δα||={float(d.norm()):.4e}")


In [ ]:
@torch.no_grad()
def encoder_qk(model, batch):
    """Layer-0 QKᵀ/√d. Independent of age: Q,K come from LN(code embeddings)."""
    x = model.embedding_table[batch["code_indices"]]
    blk = model.encoder.blocks[0]
    h_in = blk.ln_attn(x) if blk.ln_attn is not None else x
    attn = blk.attn
    bsz, length, _ = h_in.shape
    q = attn.mlp_q(h_in).view(bsz, length, attn.n_heads, attn.d_head).transpose(1, 2)
    k = attn.mlp_k(h_in).view(bsz, length, attn.n_heads, attn.d_head).transpose(1, 2)
    qk = torch.matmul(q, k.transpose(-1, -2)) * attn.scale
    pair = build_pair_mask(batch["attention_mask"])
    return qk, pair, attn


@torch.no_grad()
def encoder_logw_attn(attn_mod, tau, age_years, qk, pair):
    log_w = attn_mod.kernel(tau, attn_mod.alpha(age_years), count=False)
    pair_h = pair.unsqueeze(1)
    scores = (qk + log_w.unsqueeze(1)).masked_fill(~pair_h, float("-inf"))
    A = torch.softmax(scores, dim=-1).masked_fill(~pair_h, 0.0)
    return log_w, A


def _tv(p, q):
    if p.dim() == 4:
        p, q = p.mean(1), q.mean(1)
    return 0.5 * (p - q).abs().sum(-1)


def _entropy(A):
    if A.dim() == 4:
        A = A.mean(1)
    p = A.clamp_min(1e-12)
    return -(p * p.log()).sum(-1)


@torch.no_grad()
def sweep(model, batches, ages):
    """Counterfactual age sweep. Events/timestamps/demographics fixed."""
    ref_age = ages[0]
    per_age = {a: {"logw_abs": [], "qk_abs": [], "attn_ent": [],
                   "pool_logw_abs": [], "h": [], "logit_mean": [], "logit_std": []}
               for a in ages}
    vs_ref = {a: {"dlogw": [], "dqk": [], "tv": [], "rel_dh": [], "cos_h": [],
                  "dlogit_mean": [], "dlogit_max": [], "R_kernel_over_qk": []}
              for a in ages if a != ref_age}
    true_last_age = []
    n_seen = 0
    example = None

    for batch in batches:
        batch = to_device(batch)
        mask = batch["attention_mask"]
        lengths = batch["lengths"]
        rows = torch.arange(lengths.shape[0], device=device)
        true_last_age.append(batch["age_years"][rows, lengths - 1].cpu())
        tau, _ = tau_from_timestamps(batch["timestamps_days"], mask, lengths)
        qk, pair, attn_mod = encoder_qk(model, batch)

        cache = {}
        for a in ages:
            b_a = broadcast_age(batch, a)
            log_w, A = encoder_logw_attn(attn_mod, tau, b_a["age_years"], qk, pair)
            out = model(b_a, need_diagnostics=True)
            cache[a] = {
                "log_w": log_w, "A": A, "h": out["h"],
                "logits": out["code_logits"],
                "pool_log_w": out["pool_log_w"],
            }
            per_age[a]["logw_abs"].append(log_w[pair].abs().mean().cpu())
            per_age[a]["qk_abs"].append(qk[pair.unsqueeze(1)].abs().mean().cpu())
            per_age[a]["attn_ent"].append(_entropy(A)[mask].mean().cpu())
            per_age[a]["pool_logw_abs"].append(out["pool_log_w"][mask].abs().mean().cpu())
            per_age[a]["h"].append(out["h"].cpu())
            per_age[a]["logit_mean"].append(out["code_logits"].mean().cpu())
            per_age[a]["logit_std"].append(out["code_logits"].std(unbiased=False).cpu())

        if example is None:
            i = 0
            Li = int(lengths[0])
            example = {
                "length": Li,
                "true_age_last": float(batch["age_years"][0, lengths[0] - 1]),
                "by_age": {
                    a: {
                        "log_w_row0": cache[a]["log_w"][0, Li - 1, :Li].cpu(),
                        "qk_row0": qk[0, 0, Li - 1, :Li].cpu(),
                        "A_row0": cache[a]["A"][0, 0, Li - 1, :Li].cpu(),
                        "pool_log_w": cache[a]["pool_log_w"][0, :Li].cpu(),
                    }
                    for a in ages
                },
            }

        ref = cache[ref_age]
        qk_scale = qk[pair.unsqueeze(1)].std(unbiased=False).clamp_min(1e-8)
        for a in ages:
            if a == ref_age:
                continue
            alt = cache[a]
            vs_ref[a]["dlogw"].append((alt["log_w"] - ref["log_w"])[pair].abs().mean().cpu())
            vs_ref[a]["dqk"].append(torch.zeros(()))  # QK computed once; identically 0
            vs_ref[a]["tv"].append(_tv(ref["A"], alt["A"])[mask].mean().cpu())
            dh = alt["h"] - ref["h"]
            vs_ref[a]["rel_dh"].append(
                (dh.norm(dim=-1) / ref["h"].norm(dim=-1).clamp_min(1e-12)).mean().cpu()
            )
            vs_ref[a]["cos_h"].append(F.cosine_similarity(ref["h"], alt["h"], dim=-1).mean().cpu())
            dlogit = (alt["logits"] - ref["logits"]).abs()
            vs_ref[a]["dlogit_mean"].append(dlogit.mean().cpu())
            vs_ref[a]["dlogit_max"].append(dlogit.max().cpu())
            vs_ref[a]["R_kernel_over_qk"].append(
                ((alt["log_w"] - ref["log_w"])[pair].std(unbiased=False) / qk_scale).cpu()
            )
        n_seen += int(lengths.shape[0])
        del cache
        print(f"  processed {n_seen}/{N_SEQ}", flush=True)

    def _cat_mean(xs):
        t = torch.stack([x if x.ndim == 0 else x.mean() for x in xs])
        return float(t.mean())

    summary = {"n": n_seen, "true_age_last": torch.cat(true_last_age).numpy(), "per_age": {}, "vs_ref": {}}
    for a in ages:
        summary["per_age"][a] = {
            "logw_absmean": _cat_mean(per_age[a]["logw_abs"]),
            "qk_absmean": _cat_mean(per_age[a]["qk_abs"]),
            "attn_entropy": _cat_mean(per_age[a]["attn_ent"]),
            "pool_logw_absmean": _cat_mean(per_age[a]["pool_logw_abs"]),
            "h": torch.cat(per_age[a]["h"], dim=0),
            "logit_mean": _cat_mean(per_age[a]["logit_mean"]),
            "logit_std": _cat_mean(per_age[a]["logit_std"]),
        }
    for a, rec in vs_ref.items():
        summary["vs_ref"][a] = {k: _cat_mean(v) for k, v in rec.items()}
    summary["example"] = example
    return summary


batches = take_sequences(N_SEQ)
print(f"running counterfactual sweep on {sum(b['lengths'].shape[0] for b in batches)} sequences, ages={AGES}")
ckpt1_sweep = sweep(model, batches, AGES)
ta = ckpt1_sweep["true_age_last"]
print(f"true last-event age of these windows: min={ta.min():.1f}  median={np.median(ta):.1f}  max={ta.max():.1f}")


In [ ]:
# Quantities saved at every sweep age (means over valid pairs / sequences)
print(f"{'a':>6}  {'||Δα||_enc':>12}  {'||Δα||_pool':>12}  {'|log w|':>10}  "
      f"{'|QK/√d|':>10}  {'H(A)':>8}  {'logit μ':>10}")
enc_da = ckpt1_alpha["encoder_layer0"]["delta_alpha"]
pool_da = ckpt1_alpha["pooling"]["delta_alpha"]
for i, a in enumerate(AGES):
    s = ckpt1_sweep["per_age"][a]
    print(f"{a:6.1f}  {float(enc_da[i].norm()):12.4e}  {float(pool_da[i].norm()):12.4e}  "
          f"{s['logw_absmean']:10.4e}  {s['qk_absmean']:10.4e}  {s['attn_entropy']:8.4f}  "
          f"{s['logit_mean']:10.4f}")

print(f"\nPairwise change vs a={AGES[0]:.0f} (events/timestamps/demographics fixed)")
print(f"{'a':>6}  {'mean|Δ log w|':>14}  {'mean TV(A)':>12}  {'rel ||Δh||':>12}  "
      f"{'cos h':>8}  {'mean|Δlogit|':>14}  {'max|Δlogit|':>12}  {'σ(Δlogw)/σ(QK)':>14}")
for a in AGES[1:]:
    r = ckpt1_sweep["vs_ref"][a]
    print(f"{a:6.1f}  {r['dlogw']:14.4e}  {r['tv']:12.4e}  {r['rel_dh']:12.4e}  "
          f"{r['cos_h']:8.5f}  {r['dlogit_mean']:14.4e}  {r['dlogit_max']:12.4e}  "
          f"{r['R_kernel_over_qk']:14.4e}")

da_ped = max(curve_span(v["delta_alpha"]) for v in ckpt1_alpha.values())
da_adult = max(curve_span(site_curve(site, ADULT_AGES)[0]) for _, site in model.kernel_sites())
dlogw = max(r["dlogw"] for r in ckpt1_sweep["vs_ref"].values())
tv = max(r["tv"] for r in ckpt1_sweep["vs_ref"].values())
rel_dh = max(r["rel_dh"] for r in ckpt1_sweep["vs_ref"].values())
dlogit = max(r["dlogit_max"] for r in ckpt1_sweep["vs_ref"].values())
R = max(r["R_kernel_over_qk"] for r in ckpt1_sweep["vs_ref"].values())
qk_moved = max(abs(ckpt1_sweep["per_age"][a]["qk_absmean"] - ckpt1_sweep["per_age"][AGES[0]]["qk_absmean"])
               for a in AGES)

# Walk the markdown table top-to-bottom; first failing link is the localization.
EPS = 1e-3
steps = [
    ("Δα(a) barely changes", da_ped < EPS,
     f"pediatric span={da_ped:.4e}  adult span={da_adult:.4e}",
     "Age encoder / Fourier / MLP never learned age"),
    ("Δα changes, log w doesn't", dlogw < EPS,
     f"mean|Δ log w|={dlogw:.4e}",
     "Temporal basis or τ scaling is killing it"),
    ("log w changes, attention barely changes", tv < EPS,
     f"mean TV(A)={tv:.4e}  σ(Δlogw)/σ(QK)={R:.4e}",
     "Kernel contribution is too small relative to QK"),
    ("Attention changes, representation doesn't", rel_dh < EPS,
     f"rel ||Δh||={rel_dh:.4e}",
     "Residual/content pathway is ignoring it"),
    ("Representation changes, prediction doesn't", dlogit < EPS,
     f"max|Δlogit|={dlogit:.4e}",
     "Pretraining objective does not require the information"),
]

print("\nLocalization")
print(f"  QK age-invariance check (must be ~0): {qk_moved:.4e}")
hit = None
for obs, failed, number, cause in steps:
    flag = "FAIL" if failed else "ok  "
    print(f"  [{flag}] {obs:48s}  {number}")
    if failed and hit is None:
        hit = (obs, cause)
if hit is None:
    print("\n  Verdict: every link in the chain moves. The kernel path is not dead.")
else:
    print(f"\n  Verdict: {hit[0]}")
    print(f"  Likely problem: {hit[1]}")
if da_ped < EPS <= da_adult:
    print("  Note: Δα is flat on {1,5,10,15} but moves on adult ages.")
    print("  That is extrapolation off MIMIC support (~16.6 y+), not a dead generator.")
elif da_ped > 2.0 * max(da_adult, EPS):
    print("  Note: pediatric ||Δα|| span is much larger than the adult span.")
    print("  {1,5,10,15} is off the MIMIC support; large motion here is extrapolation,")
    print("  not evidence that a developmental kernel was learned in the pediatric range.")


In [ ]:
import matplotlib.pyplot as plt

grid = np.round(np.arange(0.0, 90.25, 0.5), 4)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

ax = axes[0]
for name, site in model.kernel_sites():
    delta, _ = site_curve(site, grid)
    ax.plot(grid, delta.norm(dim=-1).numpy(), label=name)
ax.axvspan(0, 18, color="C3", alpha=0.08, label="pediatric / off-support")
ax.set_xlabel("age (years)")
ax.set_ylabel(r"$||\Delta\alpha(a)||_2$")
ax.set_title("Age encoder (cheap; no sequences)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ex = ckpt1_sweep["example"]
for a in AGES:
    ax.plot(ex["by_age"][a]["A_row0"].numpy(), label=f"a={a:.0f}", lw=1)
ax.set_xlabel("key index (last query row)")
ax.set_ylabel(r"$A_{n j}$")
ax.set_title(f"example attention, true age={ex['true_age_last']:.1f} y, L={ex['length']}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
